# Intro to librosa — exploring an *Angry* emotion clip

A quick tour of [librosa](https://librosa.org/) for audio analysis, using one of the
**Angry** `.wav` files from our `Emotions/` dataset.

We'll cover: loading audio, the waveform, spectrograms, mel-spectrograms, MFCCs,
and a few common features used for speech-emotion recognition.

> **Setup:** `pip install librosa matplotlib numpy` (librosa pulls in soundfile/audioread for decoding).

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio

print("librosa version:", librosa.__version__)

## 1. Load an audio file

`librosa.load` returns a 1-D NumPy array of samples `y` and the sample rate `sr`.
By default librosa resamples to 22,050 Hz and mixes down to mono — pass `sr=None`
to keep the file's native sample rate.

In [ ]:
# Path is relative to the repo root. The Emotions/ folder is gitignored,
# so make sure you have the dataset locally.
audio_path = "../Emotions/Angry/03-01-05-01-01-01-01.wav"

y, sr = librosa.load(audio_path)

print(f"Samples shape : {y.shape}")
print(f"Sample rate   : {sr} Hz")
print(f"Duration      : {librosa.get_duration(y=y, sr=sr):.2f} s")

# Listen to it inline
Audio(data=y, rate=sr)

## 2. Waveform (amplitude over time)

The raw signal. Angry speech tends to be loud with sharp, high-energy bursts.

In [ ]:
plt.figure(figsize=(12, 3))
librosa.display.waveshow(y, sr=sr)
plt.title("Waveform — Angry clip")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

## 3. Spectrogram (STFT)

A spectrogram shows how the frequency content changes over time. We take the
Short-Time Fourier Transform and convert amplitude to decibels for visualization.

In [ ]:
D = librosa.stft(y)                       # complex STFT
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

plt.figure(figsize=(12, 4))
img = librosa.display.specshow(S_db, sr=sr, x_axis="time", y_axis="log")
plt.colorbar(img, format="%+2.0f dB")
plt.title("Log-frequency spectrogram")
plt.tight_layout()
plt.show()

## 4. Mel-spectrogram

The mel scale spaces frequencies the way human hearing perceives pitch. Mel-spectrograms
(and their log) are one of the most common inputs to audio deep-learning models.

In [ ]:
mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(12, 4))
img = librosa.display.specshow(mel_db, sr=sr, x_axis="time", y_axis="mel")
plt.colorbar(img, format="%+2.0f dB")
plt.title("Mel-spectrogram")
plt.tight_layout()
plt.show()

## 5. MFCCs (Mel-Frequency Cepstral Coefficients)

MFCCs are a compact summary of the spectral envelope and a classic feature for
speech-emotion recognition. Here we extract 13 coefficients per frame.

In [ ]:
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
print("MFCC shape (n_mfcc, frames):", mfccs.shape)

plt.figure(figsize=(12, 4))
img = librosa.display.specshow(mfccs, sr=sr, x_axis="time")
plt.colorbar(img)
plt.title("MFCCs")
plt.ylabel("MFCC coefficient")
plt.tight_layout()
plt.show()

## 6. A few more common features

- **Zero-crossing rate** — how often the signal changes sign (noisiness / percussiveness).
- **Spectral centroid** — the "center of mass" of the spectrum (perceived brightness).
- **RMS energy** — loudness over time.

In [ ]:
zcr      = librosa.feature.zero_crossing_rate(y)[0]
centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
rms      = librosa.feature.rms(y=y)[0]

print(f"Mean zero-crossing rate : {zcr.mean():.4f}")
print(f"Mean spectral centroid  : {centroid.mean():.1f} Hz")
print(f"Mean RMS energy         : {rms.mean():.4f}")

frames = range(len(rms))
t = librosa.frames_to_time(frames, sr=sr)

plt.figure(figsize=(12, 3))
plt.plot(t, rms, label="RMS energy")
plt.title("RMS energy over time")
plt.xlabel("Time (s)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Wrap it in a feature-extraction function

For modeling you typically reduce each clip to a fixed-length vector by averaging
features across time. This is a handy starting point for a classifier.

In [ ]:
def extract_features(path, sr=22050, n_mfcc=13):
    """Load a wav file and return a 1-D feature vector (time-averaged)."""
    y, sr = librosa.load(path, sr=sr)
    mfccs    = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    chroma   = librosa.feature.chroma_stft(y=y, sr=sr)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    zcr      = librosa.feature.zero_crossing_rate(y)
    rms      = librosa.feature.rms(y=y)
    return np.concatenate([
        mfccs.mean(axis=1),
        chroma.mean(axis=1),
        contrast.mean(axis=1),
        zcr.mean(axis=1),
        rms.mean(axis=1),
    ])

features = extract_features(audio_path)
print("Feature vector length:", features.shape[0])
print(features)

### Next steps

- Loop over every file in `Emotions/<label>/` to build a feature matrix `X` and labels `y`.
- Train a baseline classifier (e.g. `sklearn`'s SVM or RandomForest) on the averaged features.
- Or feed mel-spectrograms directly into a CNN.

See the librosa docs for more: https://librosa.org/doc/latest/index.html